<a href="https://colab.research.google.com/github/JillianneKateLBocol/Final-Project-CS2/blob/main/Kamia_Ecommerce_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
with open("/content/Ecommerce.JSON") as f:
  orders = json.load(f)
print(orders)

[{'order_id': 1, 'customer': 'Alice Brown', 'order_date': '2025-08-01', 'status': 'Delivered', 'payment_method': 'Credit Card', 'shipping_address': '123 Main St, Quezon City', 'items': [{'product': 'Headphones', 'category': 'Electronics', 'price': 1500, 'quantity': 1}, {'product': 'USB-C Cable', 'category': 'Accessories', 'price': 250, 'quantity': 2}], 'total_amount': 2000}, {'order_id': 2, 'customer': 'Bob White', 'order_date': '2025-08-03', 'status': 'Pending', 'payment_method': 'Cash on Delivery', 'shipping_address': '45 Mabini St, Davao City', 'items': [{'product': 'Smartphone', 'category': 'Electronics', 'price': 22000, 'quantity': 1}, {'product': 'Phone Case', 'category': 'Accessories', 'price': 500, 'quantity': 1}], 'total_amount': 22500}, {'order_id': 3, 'customer': 'Charlie Green', 'order_date': '2025-08-05', 'status': 'Delivered', 'payment_method': 'GCash', 'shipping_address': '78 Rizal Ave, Cebu City', 'items': [{'product': 'Book', 'category': 'Books', 'price': 450, 'quantit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials

path_to_key = "/content/Firebase.JSON"

if not firebase_admin._apps:
    cred = credentials.Certificate(path_to_key)
    firebase_admin.initialize_app(cred, {
        'databaseURL': "https://kamia-ecommerce-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully!")
else:
    print("Firebase is already connected.")

Firebase is already connected.


In [ ]:
from firebase_admin import db

ref = db.reference("orders")

for order in orders:
    ref.child(str(order["order_id"])).set(order)

print("Data uploaded successfully!")

Data uploaded successfully!


In [ ]:
ref = db.reference("orders")
orders_data = ref.get()

In [ ]:
# FEATURE 1: Show total orders and revenue
total_orders = len(orders)
total_revenue = sum(order["total_amount"] for order in orders)
print(f"\nTotal Orders: {total_orders}")
print(f"Total Revenue: {total_revenue}")


Total Orders: 4
Total Revenue: 29650


In [ ]:
ref = db.reference("orders")

while True:
    print("\n===== ECOMMERCE ORDER DATABASE =====")
    print("1. Display All Orders")
    print("2. Add New Order")
    print("3. Update Order")
    print("4. Delete Order")
    print("5. Exit")

    choice = input("Enter choice: ")

# 1. Display
    if choice == "1":
        orders = ref.get()
        if orders:
            print("\n--- Current Orders ---")

            if isinstance(orders, list):
                for index, data in enumerate(orders):
                    if data:
                        print(f"ID: {index} | Customer: {data['customer']} | Total: ₱{data['total_amount']} | Status: {data['status']}")

            else:
                for oid, data in orders.items():
                    print(f"ID: {oid} | Customer: {data['customer']} | Total: ₱{data['total_amount']} | Status: {data['status']}")
        else:
            print("\nNo orders found in the database.")

    # 2. Add
    elif choice == "2":
        oid = input("Enter Order ID: ")
        customer = input("Enter Customer Name: ")
        total = float(input("Enter Total Amount: "))
        status = "Pending"

        order_data = {
            "order_id": int(oid),
            "customer": customer,
            "total_amount": total,
            "status": status,
            "payment_method": input("Enter Payment Method: "),
            "shipping_address": input("Enter Shipping Address: "),
            "items": []
        }

        ref.child(oid).set(order_data)
        print(f"Order #{oid} for {customer} has been created!")

    # 3. Update
    elif choice == "3":
        oid = input("Enter Order ID to update: ")
        order_ref = ref.child(oid)
        order_data = order_ref.get()

        if order_data:
            print(f"\nEditing Order #{oid} for {order_data['customer']}")
            print("Current Items:")
            for i, item in enumerate(order_data['items']):
                print(f"{i}. {item['product']} (Qty: {item['quantity']}, Price: ₱{item['price']})")

            item_index = int(input("\nEnter the item number to update: "))

            if 0 <= item_index < len(order_data['items']):
                new_qty = int(input("Enter new quantity: "))
                new_price = int(input("Enter new price: "))

                order_data['items'][item_index]['quantity'] = new_qty
                order_data['items'][item_index]['price'] = new_price

                new_total = sum(item['price'] * item['quantity'] for item in order_data['items'])

                order_ref.update({
                    "items": order_data['items'],
                    "total_amount": new_total
                })

                print(f"\nProduct updated! New Order Total: ₱{new_total}")
            else:
                print("Invalid item number.")
        else:
            print("Order ID not found.")

    # 4. Delete
    elif choice == "4":
        oid = input("Enter Order ID to delete: ")

        if ref.child(oid).get():
            ref.child(oid).delete()
            print(f"Order #{oid} has been deleted.")
        else:
            print("Order ID not found.")

    # 5. Exit
    elif choice == "5":
        print("Exiting System...")
        break

    else:
        print("Invalid choice. Please try again.")

"""
Loop → keeps menu running

Conditionals (if-elif-else) → handles choices
CRUD operations:
Create → set()
Read → get()
Update → update()
Delete → delete()
Dictionary (JSON) → data structure
Firebase interaction
"""


===== ECOMMERCE ORDER DATABASE =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Exit
Enter choice: 4
Enter Order ID to delete: 6
Order #6 has been deleted.

===== ECOMMERCE ORDER DATABASE =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Exit
Enter choice: 1

--- Current Orders ---
ID: 1 | Customer: Alice Brown | Total: ₱2000 | Status: Delivered
ID: 2 | Customer: Bob White | Total: ₱22500 | Status: Pending
ID: 3 | Customer: Charlie Green | Total: ₱1050 | Status: Delivered
ID: 4 | Customer: Diana Lopez | Total: ₱4100 | Status: Shipped
ID: 5 | Customer: Jillianne | Total: ₱6767.0 | Status: Pending

===== ECOMMERCE ORDER DATABASE =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Exit
Enter choice: 2
Enter Order ID: 6
Enter Customer Name: Jillianne
Enter Total Amount: 6767
Enter Payment Method: GCash
Enter Shipping Address: PSHS-DRCDC
Order #6 for Jillianne has been created!

===== ECOMMERCE ORDER D

'\nLoop → keeps menu running\n\nConditionals (if-elif-else) → handles choices\nCRUD operations:\nCreate → set()\nRead → get()\nUpdate → update()\nDelete → delete()\nDictionary (JSON) → data structure\nFirebase interaction\n'